In [1]:
# Cell 1 — pull UniRec-0.1B base (.pth + config + tokenizer) from HuggingFace
from huggingface_hub import snapshot_download
from pathlib import Path

MODEL_DIR = snapshot_download(
    repo_id="topdu/unirec-0.1b",
    local_dir="/kaggle/working/unirec-0.1b",
    # files in this repo: config.json, generation_config.json, model.pth,
    # tokenizer.json, tokenizer_config.json, special_tokens_map.json, README.md
)

p = Path(MODEL_DIR)
print("downloaded to:", p)
for f in sorted(p.iterdir()):
    print(f"  {f.name:28} {f.stat().st_size/1e6:8.1f} MB")

pth = p / "model.pth"
assert pth.exists(), "model.pth missing!"
print(f"\nmodel.pth OK -> {pth}  ({pth.stat().st_size/1e6:.1f} MB, expect ~511 MB)")

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

downloaded to: /kaggle/working/unirec-0.1b
  .cache                            0.0 MB
  .gitattributes                    0.0 MB
  README.md                         0.0 MB
  config.json                       0.0 MB
  demo_imgs                         0.0 MB
  generation_config.json            0.0 MB
  model.pth                       535.9 MB
  special_tokens_map.json           0.1 MB
  tokenizer.json                    9.5 MB
  tokenizer_config.json             8.6 MB

model.pth OK -> /kaggle/working/unirec-0.1b/model.pth  (535.9 MB, expect ~511 MB)


In [2]:
# Cell 2 — pin the stack UniRec needs, then clone the OpenOCR source
!pip -q install "transformers==4.49.0" "tokenizers==0.21.0"
!git clone -q https://github.com/Topdu/OpenOCR /kaggle/working/OpenOCR
print("OpenOCR cloned ->", __import__("os").path.isdir("/kaggle/working/OpenOCR"))
print("\n⚠️  RESTART THE KERNEL NOW (Run → Restart & clear) before Cell 3,")
print("    so the pinned transformers/tokenizers actually take effect.")
print("    model.pth is on disk (/kaggle/working) so it survives the restart.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.1 MB/s eta 0:00:00
OpenOCR cloned -> True

⚠️  RESTART THE KERNEL NOW (Run → Restart & clear) before Cell 3,
    so the pinned transformers/tokenizers actually take effect.
    model.pth is on disk (/kaggle/working) so it survives the restart.


In [3]:
# Cell 3 — INSPECT config + UniRec modeling API (read-only, no model load yet)
import sys, json, glob, re
sys.path.insert(0, "/kaggle/working/OpenOCR")
import transformers, tokenizers
print("transformers", transformers.__version__, "| tokenizers", tokenizers.__version__)
assert transformers.__version__.startswith("4.49"), "⚠️ pin not active — restart kernel first"

MODEL_DIR = "/kaggle/working/unirec-0.1b"

print("\n========== config.json ==========")
print(json.dumps(json.load(open(f"{MODEL_DIR}/config.json")), indent=1))

print("\n========== generation_config.json ==========")
print(open(f"{MODEL_DIR}/generation_config.json").read())

# locate the modeling + configuration source
mods = glob.glob("/kaggle/working/OpenOCR/**/modeling_unirec.py", recursive=True)
cfgs = glob.glob("/kaggle/working/OpenOCR/**/configuration_unirec.py", recursive=True)
print("\nmodeling_unirec.py :", mods)
print("configuration_unirec.py :", cfgs)

src = open(mods[0]).read()
print("\n========== classes & key method signatures ==========")
for i, ln in enumerate(src.splitlines()):
    s = ln.strip()
    if s.startswith("class ") or (s.startswith("def ") and any(
        k in s for k in ["forward", "generate", "__init__", "encode", "prepare_inputs", "get_encoder", "get_decoder"])):
        print(f"  {s[:130]}")

# how is the checkpoint structured? (top-level keys tell us how to load it)
import torch
sd = torch.load(f"{MODEL_DIR}/model.pth", map_location="cpu")
sd = sd.get("state_dict", sd) if isinstance(sd, dict) else sd
keys = list(sd.keys())
print(f"\n========== checkpoint: {len(keys)} tensors ==========")
print("first 8 keys:", keys[:8])
print("last 8 keys :", keys[-8:])
print("has 'decoder' keys:", any("decoder" in k for k in keys),
      "| has 'encoder' keys:", any("encoder" in k for k in keys))

transformers 4.49.0 | tokenizers 0.21.0

========== config.json ==========
{
 "activation_dropout": 0.0,
 "activation_function": "relu",
 "architectures": [
  "VLMOCRForConditionalGenerationNew"
 ],
 "attention_dropout": 0.1,
 "bos_token_id": 0,
 "d_model": 768,
 "decoder_attention_heads": 6,
 "decoder_ffn_dim": 3072,
 "decoder_layerdrop": 0.0,
 "decoder_layers": 6,
 "decoder_start_token_id": 0,
 "encoder_layerdrop": 0.0,
 "eos_token_id": 2,
 "init_std": 0.02,
 "is_encoder_decoder": true,
 "max_position_embeddings": 3072,
 "model_type": "m2m_100",
 "pad_token_id": 1,
 "scale_embedding": true,
 "torch_dtype": "float32",
 "transformers_version": "4.49.0",
 "use_cache": true,
 "vocab_size": 56371,
 "max_length": 2048,
 "label_smoothing": 0.1
}

========== generation_config.json ==========
{
  "_from_model_config": true,
  "bos_token_id": 0,
  "decoder_start_token_id": 0,
  "eos_token_id": 2,
  "pad_token_id": 1,
  "transformers_version": "4.49.0"
}


modeling_unirec.py : ['/kaggle/working

In [4]:
# Cell 4 — load Torvex-UniRec-Velox (strict), tokenizer, confirm generate API
import sys, os, glob, time, inspect, torch, numpy as np
sys.path.insert(0, "/kaggle/working/OpenOCR")
from openrec.modeling.unirec_modeling.configuration_unirec import UniRecConfig
from openrec.modeling.unirec_modeling.modeling_unirec import UniRecForConditionalGenerationNew
from transformers import PreTrainedTokenizerFast
from PIL import Image

MODEL_DIR = "/kaggle/working/unirec-0.1b"
dev = "cuda" if torch.cuda.is_available() else "cpu"

config = UniRecConfig.from_pretrained(MODEL_DIR)
model  = UniRecForConditionalGenerationNew(config)
sd = torch.load(f"{MODEL_DIR}/model.pth", map_location="cpu")
sd = sd.get("state_dict", sd) if isinstance(sd, dict) else sd
missing, unexpected = model.load_state_dict(sd, strict=False)
print("load -> missing:", len(missing), "| unexpected:", len(unexpected))
if missing:    print("  missing[:5]:", missing[:5])
if unexpected: print("  unexpected[:5]:", unexpected[:5])
model.eval().to(dev)

tok = PreTrainedTokenizerFast(
    tokenizer_file=f"{MODEL_DIR}/tokenizer.json",
    bos_token="<|bos|>", eos_token="<|eos|>", pad_token="<|pad|>", unk_token="<|unk|>")
print("tokenizer len:", len(tok), "| eos/pad/bos:", tok.eos_token_id, tok.pad_token_id, tok.bos_token_id)

print("\nmain_input_name:", getattr(model, "main_input_name", "?"))
print("forward(...) :", str(inspect.signature(model.forward))[:300])

# UniRec NaSizeResize preprocessing (max 960x1408 /64, normalize 0.5)
def preprocess(img):
    img = img.convert("RGB"); w, h = img.size; ar = w / h
    if w > 960 or h > 1408:
        nh, nw = (1408, int(1408*ar)) if 960/1408 >= ar else (int(960/ar), 960)
    else:
        nw, nh = w, h
    nw, nh = max(nw//64*64, 64), max(nh//64*64, 64)
    a = np.asarray(img.resize((nw, nh), Image.BICUBIC), np.float32)/255.0
    a = (a - 0.5)/0.5
    return torch.from_numpy(a.transpose(2, 0, 1))[None].to(dev)

imgs = sorted(glob.glob(f"{MODEL_DIR}/demo_imgs/*"))
print("\ndemo imgs:", [os.path.basename(x) for x in imgs])
pix = preprocess(Image.open(imgs[0]))
print("pixel_values:", tuple(pix.shape))

# baseline greedy generate smoke
with torch.no_grad():
    t = time.time()
    out = model.generate(pixel_values=pix, max_new_tokens=512, num_beams=1, do_sample=False)
    dt = time.time() - t
print(f"\ngreedy generate: {out.shape[1]} tokens in {dt:.2f}s")
print("decoded:", tok.decode(out[0], skip_special_tokens=True)[:200])

2026-06-28 04:18:02.901695: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782620283.133441      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782620283.196936      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782620283.717717      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782620283.717772      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782620283.717778      23 computation_placer.cc:177] computation placer alr

load -> missing: 0 | unexpected: 0
tokenizer len: 56371 | eos/pad/bos: 2 1 0

main_input_name: input_ids
forward(...) : (pixel_values: torch.Tensor, input_ids: Optional[torch.LongTensor] = None, attention_mask: Optional[torch.Tensor] = None, decoder_input_ids: Optional[torch.LongTensor] = None, length: Optional[torch.LongTensor] = None, decoder_attention_mask: Optional[torch.LongTensor] = None, head_mask: Optional[to

demo imgs: ['Snipaste_2025-04-13_21-11-35.png', 'Snipaste_2025-04-13_21-51-45.png', 'Snipaste_2025-04-14_10-32-40.png', 'Snipaste_2025-04-14_11-05-21.png', 'Snipaste_2025-04-14_11-05-58.png', 'Snipaste_2025-04-15_23-55-25.png', 'Snipaste_2025-07-21_19-48-20.png', 'Snipaste_2025-07-21_20-53-25.png', 'clipboard_visa.png']
pixel_values: (1, 3, 320, 960)

greedy generate: 327 tokens in 3.10s
decoded: kl j Ġ Ġ E k Z J c k p j e q j c k e Ġ Ċ l Z J c l p j e q j c l e Ċ Ġ kl c k c l j q j J p c k c l j q j J p c k c l


In [5]:
# Cell 5 — proper UniRec decode (Ġ->space, Ċ->newline) + scan all demo images
id2tok = {v: k for k, v in tok.get_vocab().items()}
SPECIAL = {tok.bos_token_id, tok.eos_token_id, tok.pad_token_id}

def unirec_decode(ids):
    s = "".join(id2tok.get(int(t), "") for t in ids if int(t) not in SPECIAL)
    return s.replace("Ġ", " ").replace("Ċ", "\n").strip()

results = []
for path in imgs:
    pix = preprocess(Image.open(path))
    with torch.no_grad():
        out = model.generate(pixel_values=pix, max_new_tokens=1024, num_beams=1, do_sample=False)
    txt = unirec_decode(out[0])
    looks_latex = "\\" in txt          # real latex has backslash commands
    results.append((os.path.basename(path), out.shape[1], looks_latex, txt))
    print(f"\n=== {os.path.basename(path)}  ({out.shape[1]} tok, latex={looks_latex}) ===")
    print(txt[:300])

print("\n\n--- summary: clean formula candidates (latex + not looping) ---")
for name, n, latex, _ in results:
    flag = "✅" if (latex and n < 400) else ("🔁 loop?" if n >= 400 else "—")
    print(f"  {flag}  {name:38} {n:4d} tok  latex={latex}")


=== Snipaste_2025-04-13_21-11-35.png  (327 tok, latex=True) ===
\[\omega_{kl}^{(j)} = \mathrm{E}\left\{\pi_{k}(Z)-\left[\sum_{\ell_{1}=1}^{J}\frac{c_{1k}p_{\ell_{1}}}{\sigma_{\ell_{1}}}\right]\frac{\sigma_{j}e}{q_{j}}-\frac{c_{2k}}{2}\left(e^{2}-1\right)\right\}\] (10)
\[\left\{\pi_{l}(Z)-\left[\sum_{\ell_{2}=1}^{J}\frac{c_{1l}p_{\ell_{2}}}{\sigma_{\ell_{2}}}\ri

=== Snipaste_2025-04-13_21-51-45.png  (171 tok, latex=False) ===
SVTRv2: CTC Beats Encoder-Decoder Models in Scene Text Recognition

Yongkun Du1, Zhineng Chen1*, Hongtao Xie2, Caiyan Jia3, Yu-Gang Jiang1

1School of Computer Science, Fudan University, China2School of Information Science and Technology, USTC, China3School of Computer Science and Technology, Beijin

=== Snipaste_2025-04-14_10-32-40.png  (356 tok, latex=True) ===
Explicit evaluations of these coefficients follow from (5.12) and (5.16) and are given by 



\[gf_{m}=\begin{pmatrix}\frac{-s_{\infty}/s_{0}t^{m+\sigma+1/4}q^{2\sigma-1/4}}{(q^{2\sigma};q)_{m+1/2}^{2}(

In [6]:
# Cell 6 — Velox A/B: greedy vs n-gram self-speculative, timed + lossless verify
import torch, time

def timed_generate(pix, use_spec, K=10, reps=3):
    kw = dict(pixel_values=pix, max_new_tokens=1024, num_beams=1, do_sample=False)
    if use_spec:
        kw["prompt_lookup_num_tokens"] = K          # n-gram self-speculative
    with torch.no_grad():                            # warmup (kernel compile)
        out = model.generate(**kw)
    torch.cuda.synchronize()
    ts = []
    for _ in range(reps):
        torch.cuda.synchronize(); t = time.time()
        with torch.no_grad():
            out = model.generate(**kw)
        torch.cuda.synchronize(); ts.append(time.time() - t)
    return out, min(ts)                              # min = least noisy

print(f"{'image':40} {'tok':>4} {'greedy':>8} {'velox':>8} {'speedup':>8} {'lossless':>8}")
tot_g = tot_v = 0
for path in imgs:
    name = os.path.basename(path)
    pix = preprocess(Image.open(path))
    og, tg = timed_generate(pix, False)
    ov, tv = timed_generate(pix, True, K=10)
    ident = torch.equal(og, ov)                      # MUST be True (lossless)
    tot_g += tg; tot_v += tv
    print(f"{name:40} {og.shape[1]:4d} {tg:8.3f} {tv:8.3f} {tg/tv:7.2f}x {str(ident):>8}")

print(f"\nAGGREGATE: greedy {tot_g:.2f}s -> velox {tot_v:.2f}s  =  {tot_g/tot_v:.2f}x overall")

image                                     tok   greedy    velox  speedup lossless
Snipaste_2025-04-13_21-11-35.png          327    2.046    1.345    1.52x     True
Snipaste_2025-04-13_21-51-45.png          171    1.070    1.098    0.97x     True
Snipaste_2025-04-14_10-32-40.png          356    2.195    1.392    1.58x     True
Snipaste_2025-04-14_11-05-21.png          408    2.528    1.567    1.61x     True
Snipaste_2025-04-14_11-05-58.png          709    4.292    2.295    1.87x     True
Snipaste_2025-04-15_23-55-25.png          219    1.374    0.837    1.64x     True
Snipaste_2025-07-21_19-48-20.png          647    4.135    3.763    1.10x     True
Snipaste_2025-07-21_20-53-25.png          852    5.341    3.113    1.72x     True
clipboard_visa.png                        538    3.299    3.346    0.99x     True

AGGREGATE: greedy 26.28s -> velox 18.76s  =  1.40x overall


In [7]:
# Cell 7 — sweep draft length K on formula-heavy images
sweep = ["Snipaste_2025-04-14_11-05-58.png",  # 709 tok, dense
         "Snipaste_2025-04-14_10-32-40.png",  # 356 tok
         "Snipaste_2025-04-15_23-55-25.png"]  # 219 tok derivation
for K in [4, 6, 8, 10, 12, 16, 20]:
    sp = []
    for name in sweep:
        pix = preprocess(Image.open(f"{MODEL_DIR}/demo_imgs/{name}"))
        og, tg = timed_generate(pix, False, reps=2)
        ov, tv = timed_generate(pix, True, K=K, reps=2)
        assert torch.equal(og, ov)
        sp.append(tg / tv)
    print(f"K={K:2d}: {[f'{x:.2f}' for x in sp]}  mean {sum(sp)/len(sp):.2f}x")

K= 4: ['1.51', '1.39', '1.38']  mean 1.43x
K= 6: ['1.57', '1.43', '1.44']  mean 1.48x
K= 8: ['1.78', '1.58', '1.54']  mean 1.63x
K=10: ['1.78', '1.44', '1.65']  mean 1.62x
K=12: ['1.69', '1.34', '1.59']  mean 1.54x
K=16: ['1.88', '1.42', '1.63']  mean 1.64x
K=20: ['1.86', '1.45', '1.63']  mean 1.64x


In [8]:
# Cell 8 — multi-token decoder wrapper + PyTorch parity check (foundation of the ONNX port)
import torch
dec     = model.get_decoder()
lm_head = model.lm_head
bos     = config.decoder_start_token_id

# real encoder states from one formula image
pix = preprocess(Image.open(f"{MODEL_DIR}/demo_imgs/Snipaste_2025-04-15_23-55-25.png"))
with torch.no_grad():
    enc = model.get_encoder()(pixel_values=pix)
enc_hidden = enc.last_hidden_state if hasattr(enc, "last_hidden_state") else enc[0]
print("encoder hidden:", tuple(enc_hidden.shape))

def dec_step(input_ids, past, enc_hidden):
    # HF M2M100 decoder computes causal mask + sinusoidal positions internally
    with torch.no_grad():
        out = dec(input_ids=input_ids, encoder_hidden_states=enc_hidden,
                  past_key_values=past, use_cache=True, return_dict=True)
    return lm_head(out.last_hidden_state), out.past_key_values

# --- SEQUENTIAL: 3 single-token steps (the current greedy path) ---
ids, past, seq_logits, seq_inputs = torch.tensor([[bos]], device=dev), None, [], [bos]
for _ in range(3):
    lg, past = dec_step(ids, past, enc_hidden)
    seq_logits.append(lg[0, -1].clone())
    nxt = int(lg[0, -1].argmax()); seq_inputs.append(nxt)
    ids = torch.tensor([[nxt]], device=dev)

# --- MULTI-TOKEN: feed [bos, t1, t2] in ONE pass, no past ---
multi_ids = torch.tensor([seq_inputs[:3]], device=dev)
lg_multi, _ = dec_step(multi_ids, None, enc_hidden)
print("multi logits:", tuple(lg_multi.shape), "(expect [1, 3, vocab])\n")

ok = True
for i in range(3):
    d = (lg_multi[0, i] - seq_logits[i]).abs().max().item()
    ok &= d < 1e-3
    print(f"  pos{i}: max|logit diff| = {d:.3e}")
print("\nPARITY:", "✅ PASS — multi-token causal mask correct in PyTorch (export is safe)"
      if ok else "❌ FAIL — wrapper needs fixing before export")

encoder hidden: (1, 120, 768)
multi logits: (1, 3, 56371) (expect [1, 3, vocab])

  pos0: max|logit diff| = 1.717e-05
  pos1: max|logit diff| = 9.537e-06
  pos2: max|logit diff| = 1.144e-05

PARITY: ✅ PASS — multi-token causal mask correct in PyTorch (export is safe)


In [9]:
# Cell 8.5 (fix) — onnxruntime-gpu grabbed a CUDA-13 build; Kaggle is CUDA-12.
# CPU build is fine for the parity check (correctness, not speed).
!pip -q uninstall -y onnxruntime-gpu onnxruntime
!pip -q install onnxruntime
import onnxruntime as ort
print("onnxruntime", ort.__version__, "| providers:", ort.get_available_providers())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 74.6 MB/s eta 0:00:00
onnxruntime 1.27.0 | providers: ['AzureExecutionProvider', 'CPUExecutionProvider']


In [10]:
# Cell 9 — export multi-token decoder to ONNX (causal mask preserved) + ONNX parity
import torch, onnxruntime as ort, numpy as np

nl = config.decoder_layers
nh = config.decoder_attention_heads
hd = config.d_model // nh
OUT = "/kaggle/working/unirec_decoder_velox.onnx"

class DecExport(torch.nn.Module):
    def __init__(self, m):
        super().__init__(); self.dec = m.get_decoder(); self.lm_head = m.lm_head
    def forward(self, input_ids, encoder_hidden_states, *self_past):
        past = []
        for i in range(nl):
            ea = self.dec.layers[i].encoder_attn
            B, enc, _ = encoder_hidden_states.shape
            ck = ea.k_proj(encoder_hidden_states).view(B, enc, nh, hd).transpose(1, 2).contiguous()
            cv = ea.v_proj(encoder_hidden_states).view(B, enc, nh, hd).transpose(1, 2).contiguous()
            past.append((self_past[2*i], self_past[2*i+1], ck, cv))   # (self_k, self_v, cross_k, cross_v)
        out = self.dec(input_ids=input_ids, encoder_hidden_states=encoder_hidden_states,
                       past_key_values=tuple(past), use_cache=True, return_dict=True)
        logits = self.lm_head(out.last_hidden_state)
        present = []
        for lp in out.past_key_values:
            present += [lp[0], lp[1]]                                  # keep self KV only
        return (logits, *present)

wrap = DecExport(model).eval()

# example inputs (K=3 query tokens, past_len=4 just for tracing; dynamic at runtime)
ex_ids  = torch.tensor([seq_inputs[:3]], device=dev)
ex_past = [torch.zeros(1, nh, 4, hd, device=dev) for _ in range(2*nl)]
args = (ex_ids, enc_hidden, *ex_past)

in_names  = ["input_ids", "encoder_hidden_states"] + \
            [f"past_self_{'kv'[j%2]}_{j//2}" for j in range(2*nl)]
out_names = ["logits"] + [f"present_self_{'kv'[j%2]}_{j//2}" for j in range(2*nl)]
dyn = {"input_ids": {1: "K"}, "encoder_hidden_states": {1: "enc"}, "logits": {1: "K"}}
for n in in_names[2:]:  dyn[n] = {2: "past"}
for n in out_names[1:]: dyn[n] = {2: "tot"}

torch.onnx.export(wrap, tuple(args), OUT, input_names=in_names, output_names=out_names,
                  dynamic_axes=dyn, opset_version=17, do_constant_folding=True,
                  dynamo=False)          # <-- legacy TorchScript exporter (no onnxscript needed)
print("exported ->", OUT)

# ---- ONNX parity: multi-token [bos,t1,t2], empty past, vs PyTorch sequential logits ----
sess = ort.InferenceSession(OUT, providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
feeds = {"input_ids": np.array([seq_inputs[:3]], np.int64),
         "encoder_hidden_states": enc_hidden.cpu().numpy()}
for j in range(2*nl):
    feeds[f"past_self_{'kv'[j%2]}_{j//2}"] = np.zeros((1, nh, 0, hd), np.float32)
onnx_logits = sess.run(None, feeds)[0]   # [1, 3, vocab]
print("onnx logits:", onnx_logits.shape)
# fix the comparison only (ONNX already ran; seq_logits/onnx_logits are in memory)
ok = True
for i in range(3):
    d = np.abs(onnx_logits[0, i] - seq_logits[i].detach().cpu().numpy()).max()
    ok &= d < 1e-2
    print(f"  pos{i}: max|logit diff vs PyTorch| = {d:.3e}")
print("\nONNX PARITY:", "✅ PASS — multi-token verify works in ONNX, Velox port unblocked"
      if ok else "❌ FAIL — traced constants didn't generalize, we re-export")

/tmp/ipykernel_23/3223585222.py:42: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(wrap, tuple(args), OUT, input_names=in_names, output_names=out_names,
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:88: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if input_shape[-1] > 1 or self.sliding_window is not None:
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:16

exported -> /kaggle/working/unirec_decoder_velox.onnx


/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:147: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


onnx logits: (1, 3, 56371)
  pos0: max|logit diff vs PyTorch| = 1.144e-05
  pos1: max|logit diff vs PyTorch| = 3.433e-05
  pos2: max|logit diff vs PyTorch| = 1.907e-05

ONNX PARITY: ✅ PASS — multi-token verify works in ONNX, Velox port unblocked


In [11]:
# Cell 10 — ONNX greedy vs ONNX n-gram speculative decode: lossless verify + speedup
import numpy as np, time
eos, bos = config.eos_token_id, config.decoder_start_token_id
PAST = [f"past_self_{'kv'[j%2]}_{j//2}" for j in range(2*nl)]

def step(input_ids, enc_np, past):
    feeds = {"input_ids": input_ids.astype(np.int64), "encoder_hidden_states": enc_np}
    for n, p in zip(PAST, past): feeds[n] = p
    o = sess.run(None, feeds)
    return o[0], o[1:]                      # logits [1,K,vocab], present self-KV
def empty():  return [np.zeros((1, nh, 0, hd), np.float32) for _ in range(2*nl)]

def onnx_greedy(enc_np, max_new=384):
    past, gen, cur = empty(), [bos], np.array([[bos]])
    for _ in range(max_new):
        lg, past = step(cur, enc_np, past)
        nxt = int(lg[0, -1].argmax()); gen.append(nxt)
        if nxt == eos: break
        cur = np.array([[nxt]])
    return gen

def lookup(gen, n=3, K=16):
    if len(gen) < n: return []
    suf = tuple(gen[-(n-1):])
    for j in range(len(gen)-(n-1)-1, -1, -1):
        if tuple(gen[j:j+n-1]) == suf: return gen[j+n-1:j+n-1+K]
    return []

def onnx_velox(enc_np, max_new=384, K=16):
    past = empty()
    lg, past = step(np.array([[bos]]), enc_np, past)       # seed 1 token
    gen = [bos, int(lg[0, -1].argmax())]
    while len(gen) < max_new and gen[-1] != eos:
        draft = lookup(gen, K=K)
        block = np.array([[gen[-1]] + draft])
        lg, present = step(block, enc_np, past)
        preds = lg[0].argmax(-1)
        accepted = [int(preds[0])]
        for i in range(len(draft)):
            if draft[i] == accepted[i]: accepted.append(int(preds[i+1]))
            else: break
        gen.extend(accepted)
        past = [p[:, :, :len(gen)-1, :] for p in present]   # KV rollback to committed
        if eos in accepted:
            gen = gen[:len(gen)-len(accepted)+accepted.index(eos)+1]; break
    return gen

def get_enc(path):
    pix = preprocess(Image.open(path))
    with torch.no_grad():
        e = model.get_encoder()(pixel_values=pix)
    return (e.last_hidden_state if hasattr(e, "last_hidden_state") else e[0]).detach().cpu().numpy()

test = ["Snipaste_2025-04-14_11-05-58.png", "Snipaste_2025-04-14_10-32-40.png",
        "Snipaste_2025-04-15_23-55-25.png"]
print(f"{'image':40} {'tok':>4} {'greedy_s':>9} {'velox_s':>9} {'speedup':>8} {'lossless':>8}")
for name in test:
    enc_np = get_enc(f"{MODEL_DIR}/demo_imgs/{name}")
    t = time.time(); g = onnx_greedy(enc_np);  tg = time.time()-t
    t = time.time(); v = onnx_velox(enc_np);   tv = time.time()-t
    print(f"{name:40} {len(g):4d} {tg:9.2f} {tv:9.2f} {tg/tv:7.2f}x {str(g==v):>8}")

image                                     tok  greedy_s   velox_s  speedup lossless
Snipaste_2025-04-14_11-05-58.png          385     18.61      6.74    2.76x    False
Snipaste_2025-04-14_10-32-40.png          356      9.37      5.98    1.57x     True
Snipaste_2025-04-15_23-55-25.png          219      5.54      3.40    1.63x     True


In [12]:
# Cell 10b — larger cap so long formulas reach EOS naturally (removes the cap-boundary mismatch)
print(f"{'image':40} {'tok':>4} {'greedy_s':>9} {'velox_s':>9} {'speedup':>8} {'lossless':>8}")
tot_g = tot_v = 0
for name in test:
    enc_np = get_enc(f"{MODEL_DIR}/demo_imgs/{name}")
    t = time.time(); g = onnx_greedy(enc_np, max_new=1024); tg = time.time()-t
    t = time.time(); v = onnx_velox(enc_np,  max_new=1024); tv = time.time()-t
    tot_g += tg; tot_v += tv
    print(f"{name:40} {len(g):4d} {tg:9.2f} {tv:9.2f} {tg/tv:7.2f}x {str(g==v):>8}")
print(f"\nAGGREGATE (CPU, conservative): {tot_g:.1f}s -> {tot_v:.1f}s = {tot_g/tot_v:.2f}x")

image                                     tok  greedy_s   velox_s  speedup lossless
Snipaste_2025-04-14_11-05-58.png          709     38.22     19.25    1.99x     True
Snipaste_2025-04-14_10-32-40.png          356     15.69      9.49    1.65x     True
Snipaste_2025-04-15_23-55-25.png          219      9.44      5.56    1.70x     True

AGGREGATE (CPU, conservative): 63.3s -> 34.3s = 1.85x
